# SHS generation + analysis

Generates a synthetic MSA with the RNA SHS generator and checks how well the
analyzer recovers the generator input from it.

Example: two hairpins joined by a linker. Stem 1 pairs with interaction 1.0,
stem 2 with 0.5, and linker base 25 forms a base triple with pair (31, 47).
`watson_crick_cov` is the one approach where every estimator applies.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
import numpy as np
import pandas as pd

sys.path.insert(0, "../../SHS-Generator-RNA")
import generator_config
import shs_generator
import shs_analyzer

pd.set_option("display.max_colwidth", None)  # show the full `detail` strings

## Parameters

In [ ]:
RNA_SEQ = "GAGCGGAUCUUCGAAGAUCUGCAACAUAGCCCUGGAGAAACUAUCUAGGACU"
# Zero-based [a, b, interaction] list, or a dot-bracket string such as
# "..(((((((......)))))))........((((((.......))))))..."
STRUCTURE = [[2, 21, 1.0], [3, 20, 1.0], [4, 19, 1.0], [5, 18, 1.0], [6, 17, 1.0], [7, 16, 1.0], [8, 15, 1.0],
             [30, 48, 0.5], [31, 47, 0.5], [32, 46, 0.5], [33, 45, 0.5], [34, 44, 0.5], [35, 43, 0.5],
             [25, 31, 0.5]]
MUTATION_RATES = None  # optional per-position list; overrides the paired / unpaired rates
SEED = 42

PARAMS = dict(
    N=2000,
    pair_mutation_approach="watson_crick_cov",  # covariance | watson_crick | watson_crick_cov | original | none
    mutation_rate_unpaired=0.15,
    mutation_rate_paired=0.25,
    # Replacement base of a mutated pair member (watson_crick, watson_crick_cov, original multiplets)
    pair_wc_prob=0.8,
    pair_wobble_prob=0.3,
    wobble_prob=0.1,  # pair mutation of basic pairs (original only)
    # Insertions / deletions, paired
    stem_single_insertion_prob=0.03,
    stem_long_insertion_prob=0.02,
    stem_single_deletion_prob=0.05,
    stem_pair_deletion_prob=0.1,
    stem_max_insertion_fraction=0.1,
    # Insertions / deletions, unpaired
    loop_single_insertion_prob=0.08,
    loop_single_deletion_prob=0.1,
    loop_long_insertion_prob=0.04,
    loop_long_deletion_prob=0.05,
    loop_max_insertion_fraction=0.15,
    loop_max_deletion_fraction=0.1,
)

## Generate the SHS

In [ ]:
parameters, paired, unpaired = generator_config.split_parameters(PARAMS)
pairs = generator_config.build_pair_map(RNA_SEQ, STRUCTURE, paired, unpaired, MUTATION_RATES)
msa = shs_generator.MsaGenerator(parameters, seed=SEED).generate(RNA_SEQ, pairs)

data = shs_analyzer.MsaData.from_rows(msa, pairs)
# To analyze an exported MSA instead: data = shs_analyzer.MsaData.from_path("<af3_input.json or .fasta>", pairs)
feat = shs_analyzer.Features(data)

print(f"{len(msa)} MSA rows for {len(RNA_SEQ)} RNA positions")
print("\n".join(msa[:11]))

## Expected vs recovered covariance

With `covariance` and `watson_crick_cov`, the later base b of a pair (a, b) mutates
more often if a mutated: cov(a, b) = I · m_b · (1 − m_a) / (k + 1), where k is the
number of earlier partners of b (1 for a basic pair). `none` and `watson_crick`
mutate pair members independently (0). `original` has no closed form and is shown as 0 too.

In [ ]:
expected = shs_analyzer.expected_covariance(data, parameters.pair_mutation_approach)
shs_analyzer.plot_covariance_comparison(data.seq, expected, feat.covariance)

## Covariance per pair

In [ ]:
_ = shs_analyzer.plot_covariance_classification(data.seq, feat.covariance, pairs=data.pairs)

## Deletions and insertions

In [ ]:
_ = shs_analyzer.plot_deletions_and_insertion(feat.col_deletion_rate, feat.col_insertion_mean, pairs=data.pairs)

In [ ]:
# Alternative: split panels. Deletion rate per position (top) and insertion rate per gap
# (bottom, share of rows with any insertion) on a shared x-axis.
_ = shs_analyzer.plot_indels(feat.col_deletion_rate, feat.col_insertion_rate, pairs=data.pairs)

## Mutation-rate comparison

In [ ]:
input_rates = np.array([pairs.mutation_rate(i) for i in range(len(data.seq))])
_, metrics = shs_analyzer.plot_mutation_rate_comparison(input_rates, feat.col_mut_rate)
print(f"MAE: {metrics['mae']:.4f} | RMSE: {metrics['rmse']:.4f} | Pearson r: {metrics['pearson_r']:.4f}")

## Recovered parameters

`pair_wc_prob` and `pair_wobble_prob` are only recovered for `watson_crick` and
`watson_crick_cov`, `interaction_strength` only for `covariance` and `watson_crick_cov`.

In [ ]:
report = shs_analyzer.analyze_data(data)
shs_analyzer.compare_parameters(report, shs_analyzer.known_parameters(data, parameters))